In [3]:
import pandas as pd

columns = [
    "engine_id",
    "cycle",
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
    "sensor_1",
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_5",
    "sensor_6",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_10",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_16",
    "sensor_17",
    "sensor_18",
    "sensor_19",
    "sensor_20",
    "sensor_21"
]

train = pd.read_csv(
    "raw/train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

test = pd.read_csv(
    "raw/test_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

print(train.shape)
print(test.shape)

(20631, 26)
(13096, 26)


In [4]:
print("Missing values in training data:")
print(train.isnull().sum())

print("\nMissing values in test data:")
print(test.isnull().sum())

Missing values in training data:
engine_id       0
cycle           0
op_setting_1    0
op_setting_2    0
op_setting_3    0
sensor_1        0
sensor_2        0
sensor_3        0
sensor_4        0
sensor_5        0
sensor_6        0
sensor_7        0
sensor_8        0
sensor_9        0
sensor_10       0
sensor_11       0
sensor_12       0
sensor_13       0
sensor_14       0
sensor_15       0
sensor_16       0
sensor_17       0
sensor_18       0
sensor_19       0
sensor_20       0
sensor_21       0
dtype: int64

Missing values in test data:
engine_id       0
cycle           0
op_setting_1    0
op_setting_2    0
op_setting_3    0
sensor_1        0
sensor_2        0
sensor_3        0
sensor_4        0
sensor_5        0
sensor_6        0
sensor_7        0
sensor_8        0
sensor_9        0
sensor_10       0
sensor_11       0
sensor_12       0
sensor_13       0
sensor_14       0
sensor_15       0
sensor_16       0
sensor_17       0
sensor_18       0
sensor_19       0
sensor_20       0
sensor

In [5]:
feature_cols = [
    col for col in train.columns
    if col not in ["engine_id", "cycle"]
]

unique_counts = train[feature_cols].nunique().sort_values()

print(unique_counts)

sensor_1           1
op_setting_3       1
sensor_5           1
sensor_10          1
sensor_18          1
sensor_19          1
sensor_16          1
sensor_6           2
sensor_17         13
op_setting_2      13
sensor_8          53
sensor_13         56
sensor_20        120
op_setting_1     158
sensor_11        159
sensor_2         310
sensor_12        427
sensor_7         513
sensor_15       1918
sensor_3        3012
sensor_4        4051
sensor_21       4745
sensor_14       6078
sensor_9        6403
dtype: int64


In [6]:
constant_features = [
    col for col in feature_cols
    if train[col].nunique() <= 1
]

print("Constant features:")
print(constant_features)

Constant features:
['op_setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']


In [7]:
std_values = train[feature_cols].std().sort_values()

print(std_values.head(10))

sensor_1        0.000000e+00
op_setting_3    0.000000e+00
sensor_10       0.000000e+00
sensor_19       0.000000e+00
sensor_18       0.000000e+00
sensor_16       3.469531e-18
sensor_5        5.329200e-15
op_setting_2    2.930621e-04
sensor_6        1.388985e-03
op_setting_1    2.187313e-03
dtype: float64


In [8]:
# Find the maximum cycle for each engine
max_cycles = train.groupby("engine_id")["cycle"].max()

# Calculate RUL
train["RUL"] = train["engine_id"].map(max_cycles) - train["cycle"]

# Check
train[["engine_id", "cycle", "RUL"]].head(10)

,engine_id,cycle,RUL
0,1,1,191
1,1,2,190
2,1,3,189
3,1,4,188
4,1,5,187
5,1,6,186
6,1,7,185
7,1,8,184
8,1,9,183
9,1,10,182


In [9]:
print(train.groupby("engine_id")["RUL"].min().head(10))

engine_id
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    0
Name: RUL, dtype: int64


In [10]:
selected_features = [
    "cycle",
    "op_setting_1",
    "op_setting_2",
    "op_setting_3",
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_17",
    "sensor_20",
    "sensor_21"
]

In [11]:
train_processed = train[
    ["engine_id"] + selected_features + ["RUL"]
].copy()

In [12]:
test_processed = test[
    ["engine_id"] + selected_features
].copy()

In [13]:
print(train_processed.head())
print(train_processed.shape)
print(train_processed.columns.tolist())

   engine_id  cycle  op_setting_1  op_setting_2  op_setting_3  sensor_2  \
0          1      1       -0.0007       -0.0004         100.0    641.82   
1          1      2        0.0019       -0.0003         100.0    642.15   
2          1      3       -0.0043        0.0003         100.0    642.35   
3          1      4        0.0007        0.0000         100.0    642.35   
4          1      5       -0.0019       -0.0002         100.0    642.37   

   sensor_3  sensor_4  sensor_7  sensor_8  sensor_9  sensor_11  sensor_12  \
0   1589.70   1400.60    554.36   2388.06   9046.19      47.47     521.66   
1   1591.82   1403.14    553.75   2388.04   9044.07      47.49     522.28   
2   1587.99   1404.20    554.26   2388.08   9052.94      47.27     522.42   
3   1582.79   1401.87    554.45   2388.11   9049.48      47.13     522.86   
4   1582.85   1406.22    554.00   2388.06   9055.15      47.28     522.19   

   sensor_13  sensor_14  sensor_15  sensor_17  sensor_20  sensor_21  RUL  
0    2388.0

In [15]:
import os

os.makedirs("processed", exist_ok=True)

In [16]:
train_processed.to_csv(
    "processed/train_FD001_processed.csv",
    index=False
)

test_processed.to_csv(
    "processed/test_FD001_processed.csv",
    index=False
)

print("Common processed files created successfully!")

Common processed files created successfully!
